# Atividade: Assistente de IA Generativa com Hugging Face e Gemini

**Disciplina:** Disruptive Architectures: IoT, IoB & Generative AI

**Grupo:**
- Nome: Moisés Barsoti Andrade de Oliveira RM565049


**Tema escolhido:** Assistente de casa inteligente

---

Neste notebook o grupo vai construir um assistente de IA em 4 etapas (e 1 bônus):

1. Assistente com personalidade (Hugging Face)
2. Comparação Hugging Face x Gemini
3. Chat com memória
4. Interface web com Gradio
5. (Bônus) API com FastAPI

Os trechos marcados com **`# >>> PERSONALIZE`** devem ser alterados pelo grupo.
Execute as células em ordem, de cima para baixo.

## 0. Configuração

In [1]:
# huggingface_hub -> cliente para chamar modelos remotamente (API do Hugging Face)
# google-genai    -> SDK oficial do Google Gemini
# gradio          -> cria interfaces web a partir de funções Python
!pip install huggingface_hub google-genai gradio -q

In [2]:
from huggingface_hub import InferenceClient
from google import genai
from google.genai import types
from google.colab import userdata

# Os tokens ficam nos Secrets do Colab (ícone de chave na barra lateral)
HF_TOKEN = userdata.get("HF_TOKEN")
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

print("HF_TOKEN ok" if HF_TOKEN else "ERRO: adicione HF_TOKEN nos Secrets do Colab")
print("GEMINI_API_KEY ok" if GEMINI_API_KEY else "ERRO: adicione GEMINI_API_KEY nos Secrets do Colab")

HF_TOKEN ok
GEMINI_API_KEY ok


In [9]:
# Modelos usados na atividade (os mesmos da Aula 05)
MODELO_HF = "meta-llama/Llama-3.1-8B-Instruct"
MODELO_GEMINI = "gemini-3.5-flash-lite" # Realizada alteração pois estava dando erro na versão antiga

cliente_hf = InferenceClient(model=MODELO_HF, token=HF_TOKEN, provider="auto")
cliente_gemini = genai.Client(api_key=GEMINI_API_KEY)

In [4]:
# Assistente de Casa Inteligente

SYSTEM_PROMPT = """Você é o SmartHome AI, um assistente especializado em casa inteligente e automação residencial.
Ajude o usuário a entender, configurar e utilizar dispositivos como sensores, lâmpadas inteligentes, tomadas conectadas, câmeras, fechaduras, assistentes de voz e sistemas de monitoramento.
Explique também conceitos de IoT aplicados à residência, como conectividade Wi-Fi, automações, sensores e integração entre dispositivos.
Responda sempre em português, de forma clara, objetiva e amigável, utilizando no máximo 5 frases.
Quando possível, dê exemplos práticos de automações residenciais.
Não invente características de dispositivos que não conhece.
Se a pergunta não tiver relação com casa inteligente, automação residencial ou IoT aplicado à residência, informe educadamente que esse assunto está fora da sua especialidade."""

---
## Etapa 1: Assistente com personalidade (Hugging Face)

Cada mensagem enviada ao modelo tem uma etiqueta `role` que diz quem escreveu:

- `system`: a regra que o assistente deve seguir
- `user`: a pergunta do usuário

Nesta etapa, a **mesma pergunta** é enviada três vezes, e **só o `system` muda**:

| Chamada | system | user |
|---|---|---|
| 1 | Especialista no tema do grupo | mesma pergunta |
| 2 | Professor para crianças | mesma pergunta |
| 3 | Resposta em uma frase | mesma pergunta |

Se as respostas saírem diferentes, a diferença veio só do `system`. É assim que se criam assistentes diferentes em cima do mesmo modelo.

In [5]:
def perguntar_hf(pergunta, system, temperatura=0.7):
    """Envia uma pergunta ao modelo do Hugging Face e devolve o texto da resposta."""
    mensagens = [
        {"role": "system", "content": system},   # como o assistente deve se comportar
        {"role": "user",   "content": pergunta}, # o que o usuário perguntou
    ]
    resposta = cliente_hf.chat_completion(
        messages=mensagens,
        max_tokens=300,
        temperature=temperatura,
    )
    return resposta.choices[0].message.content

In [6]:
# crie 3 personalidades diferentes para o assistente do grupo.
personalidades = {
    "Especialista": SYSTEM_PROMPT,

    "Professor para crianças": """
    Você é um professor que explica casa inteligente e automação residencial para crianças de 10 anos.
    Use palavras simples, exemplos do dia a dia e comparações fáceis de entender.
    Explique sensores, lâmpadas inteligentes, assistentes de voz e dispositivos conectados.
    Responda sempre em português.
    """,

    "Resumido": """
    Você é um assistente especializado em casa inteligente e automação residencial.
    Responda perguntas sobre sensores, dispositivos conectados e automações residenciais.
    Responda sempre em português usando apenas uma única frase curta.
    """
}

# uma pergunta relacionada ao tema do grupo.
pergunta = "Como um sensor de presença pode ajudar a economizar energia em casa?"

for nome, system in personalidades.items():
    print(f"===== {nome} =====")
    print(perguntar_hf(pergunta, system))
    print()

===== Especialista =====
Um sensor de presença pode ajudar a economizar energia em casa ao automatizar a iluminação e outras funções, como por exemplo:

* A iluminação de um quarto é ativada apenas quando alguém está presente, evitando que permaneça acesa durante a noite ou quando a casa está vazia.

Essa automação pode ser feita conectando o sensor de presença a uma lâmpada inteligente, que pode ser ligada e desligada automaticamente.

===== Professor para crianças =====
Que pergunta ótima!

Imagine que você está em casa e esqueceu de fechar a porta da cozinha. Agora, o sensor de presença vai ajudar a economizar energia!

O sensor de presença é um dispositivo que detecta se alguém está na sala ou não. Se a sala estiver vazia, ele pode desligar as lâmpadas ou as luzes, economizando energia. É como se ele estivesse dizendo: "Ah, ninguém está aqui, posso desligar a luz para não gastar energia desnecessariamente!"

Mas, e se você estiver fazendo um trabalho em casa e precisa da luz? O sen

**Observações do grupo (Etapa 1):**

- O que mudou nas respostas de cada personalidade?
- Qual `system` gerou a resposta mais útil para o tema? Por quê?

As respostas mudaram principalmente no nível de detalhe, na linguagem e na forma de explicar o mesmo assunto. A personalidade Especialista apresentou uma resposta mais técnica e objetiva, explicando a automação do sensor de presença com lâmpadas inteligentes. A personalidade Professor para crianças utilizou linguagem mais simples, exemplos do cotidiano e comparações para facilitar o entendimento. Já a personalidade Resumido respondeu de forma direta em apenas uma frase.
Para o tema de casa inteligente, consideramos que o system Especialista gerou a resposta mais útil, pois apresentou uma explicação mais adequada ao contexto de automação residencial, relacionando o sensor de presença com economia de energia e controle automático da iluminação.

---
## Etapa 2: Hugging Face x Gemini

Agora as mesmas perguntas vão para **dois modelos diferentes**.

No Gemini, o `system` não vai dentro da lista de mensagens. Ele é passado no parâmetro `system_instruction`.

In [7]:
def perguntar_gemini(pergunta, system):
    """Envia uma pergunta ao Gemini e devolve o texto da resposta."""
    resposta = cliente_gemini.models.generate_content(
        model=MODELO_GEMINI,
        contents=pergunta,
        config=types.GenerateContentConfig(
            system_instruction=system,  # equivalente ao role "system"
            max_output_tokens=300,
        ),
    )
    return resposta.text

In [10]:
# 3 perguntas sobre o tema.
perguntas = [
    "Quais sensores são mais usados em uma casa inteligente e para que serve cada um?",
    "Qual a diferença entre Wi-Fi e Zigbee na conexão de dispositivos de uma casa inteligente?",
    "Quais cuidados de segurança devo ter ao usar câmeras e fechaduras inteligentes conectadas à internet?",
]

for p in perguntas:
    print("PERGUNTA:", p)

    print("\n--- Hugging Face (Llama) ---")
    print(perguntar_hf(p, SYSTEM_PROMPT))

    print("\n--- Gemini ---")
    print(perguntar_gemini(p, SYSTEM_PROMPT))

    print("\n" + "=" * 60 + "\n")

PERGUNTA: Quais sensores são mais usados em uma casa inteligente e para que serve cada um?

--- Hugging Face (Llama) ---
Em uma casa inteligente, os sensores são fundamentais para a automação e segurança residencial. Aqui estão alguns dos sensores mais comuns e suas funções:

1. **Sensor de movimento**: Detecta a presença de pessoas ou animais em uma determinada área, como uma sala ou corredor. Pode ser utilizado para atividades como iluminação automática, alarmes de segurança ou alertas de presença.

2. **Sensor de temperatura e umidade**: Registra a temperatura e a umidade do ambiente, podendo ser usado para automação de climatização, monitoramento de condições de humidade ou alertas de problemas potenciais nos sistemas de aquecimento ou refrigeração.

3. **Sensor de luz**: Mede a intensidade da luz ambiente, permitindo a automação da iluminação, como acesse automaticamente as lâmpadas quando a luz natural diminui.

4. **Sensor de pressão**: Usado em sistemas de iluminação, pode cont

**Observações do grupo (Etapa 2):**

- Os dois modelos seguiram as regras do `SYSTEM_PROMPT` (idioma, tamanho, tema)?
- Qual respondeu melhor? Em qual pergunta a diferença foi maior?

Os dois modelos seguiram o tema de casa inteligente e responderam em português. Porém, o Gemini respeitou melhor o limite de tamanho definido no SYSTEM_PROMPT, apresentando respostas mais curtas e objetivas, enquanto o Hugging Face gerou respostas mais extensas em alguns casos.

De forma geral, o Gemini apresentou respostas mais claras e diretas nas três perguntas. A maior diferença foi percebida na pergunta sobre Wi-Fi e Zigbee, pois o Gemini explicou de forma mais objetiva características como rede mesh, consumo de bateria e necessidade de um hub, enquanto o Hugging Face apresentou uma explicação mais genérica. Também percebemos diferenças na seleção dos exemplos de sensores e nas recomendações de segurança para dispositivos conectados.

---
## Etapa 3: Chat com memória

O modelo **não guarda memória** entre uma chamada e outra.
Quem guarda a conversa é o nosso código, na lista `historico`, que é enviada inteira a cada mensagem.

Comandos do chat:
- `sair` encerra o chat
- `limpar` apaga o histórico (o assistente "esquece" a conversa)
- `historico` mostra quantas mensagens estão guardadas

**Teste sugerido:** diga seu nome, pergunte "qual é o meu nome?", digite `limpar` e pergunte de novo.

In [11]:
historico = [{"role": "system", "content": SYSTEM_PROMPT}]

print("Chat iniciado! Comandos: sair | limpar | historico\n")

while True:
    entrada = input("Você: ")

    if entrada.lower() == "sair":
        print("Encerrando chat.")
        break

    if entrada.lower() == "limpar":
        historico = [{"role": "system", "content": SYSTEM_PROMPT}]  # mantém só o system
        print("\n(histórico apagado)\n")
        continue

    if entrada.lower() == "historico":
        print(f"\n(mensagens no histórico: {len(historico)})\n")
        continue

    historico.append({"role": "user", "content": entrada})

    resposta = cliente_hf.chat_completion(messages=historico, max_tokens=300)
    texto = resposta.choices[0].message.content

    historico.append({"role": "assistant", "content": texto})

    print(f"\nAssistente: {texto}\n")

Chat iniciado! Comandos: sair | limpar | historico

Você: sair
Encerrando chat.


In [12]:
# Veja como ficou a lista enviada ao modelo
for msg in historico:
    print(f"[{msg['role']}] {msg['content'][:80].replace(chr(10), ' ')}")

[system] Você é o SmartHome AI, um assistente especializado em casa inteligente e automaç


**Observações do grupo (Etapa 3):**

- O que aconteceu quando vocês perguntaram o nome antes e depois do `limpar`?
- Explique, com suas palavras, por que isso acontece.

Antes de usar o comando limpar, o assistente conseguiu lembrar o nome informado anteriormente, porque essa informação ainda estava armazenada na lista historico e era enviada novamente ao modelo junto com as novas mensagens.

Depois de usar o comando limpar, o assistente não conseguiu mais informar o nome, pois o histórico da conversa foi apagado e apenas o SYSTEM_PROMPT permaneceu na lista.

Isso acontece porque o modelo não guarda memória entre uma chamada e outra. A memória é mantida pelo próprio código, através da lista historico. Quando essa lista é apagada, as mensagens anteriores deixam de ser enviadas ao modelo e, por isso, ele não tem mais acesso às informações da conversa anterior.

---
## Etapa 4: Interface web com Gradio

O assistente ganha uma interface web, com a opção de escolher o modelo (Hugging Face ou Gemini).

O Gradio entrega o histórico da conversa já no formato de `role`/`content`.
A função `texto_da_mensagem` existe porque, dependendo da versão do Gradio, o conteúdo chega como texto simples ou como lista.

In [13]:
import gradio as gr

def texto_da_mensagem(conteudo):
    """Extrai o texto de uma mensagem do histórico do Gradio."""
    if isinstance(conteudo, str):
        return conteudo
    if isinstance(conteudo, list):
        return " ".join(item.get("text", "") for item in conteudo if isinstance(item, dict))
    return str(conteudo)


def responder(mensagem, historico_gradio, provedor):
    if provedor == "Hugging Face":
        # Formato HF: roles system, user e assistant
        mensagens = [{"role": "system", "content": SYSTEM_PROMPT}]
        for msg in historico_gradio:
            mensagens.append({"role": msg["role"], "content": texto_da_mensagem(msg["content"])})
        mensagens.append({"role": "user", "content": mensagem})

        resposta = cliente_hf.chat_completion(messages=mensagens, max_tokens=300)
        return resposta.choices[0].message.content

    else:
        # Formato Gemini: roles user e model; o system vai em system_instruction
        conteudos = []
        for msg in historico_gradio:
            papel = "model" if msg["role"] == "assistant" else "user"
            conteudos.append({"role": papel, "parts": [{"text": texto_da_mensagem(msg["content"])}]})
        conteudos.append({"role": "user", "parts": [{"text": mensagem}]})

        resposta = cliente_gemini.models.generate_content(
            model=MODELO_GEMINI,
            contents=conteudos,
            config=types.GenerateContentConfig(system_instruction=SYSTEM_PROMPT, max_output_tokens=300),
        )
        return resposta.text

In [15]:
# título, descrição e exemplos de acordo com o tema.
gr.ChatInterface(
    fn=responder,
    additional_inputs=[
        gr.Radio(
            ["Hugging Face", "Gemini"],
            value="Hugging Face",
            label="Modelo"
        )
    ],
    title="SmartHome AI - Assistente de Casa Inteligente",
    description="Tire dúvidas sobre automação residencial, sensores, segurança, iluminação inteligente e dispositivos conectados.",
    examples=[
        ["Como funciona um sensor de presença em uma casa inteligente?", "Hugging Face"],
        ["Qual a diferença entre Wi-Fi e Zigbee?", "Gemini"],
        ["Como posso economizar energia usando automação residencial?", "Hugging Face"],
    ],
).launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://21e6de3bdcea1eccc9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


**Observações do grupo (Etapa 4):**

- Tire um print da interface funcionando e coloque no README do repositório do grupo.
- Troque de modelo no meio da conversa. O assistente continuou lembrando do que foi dito? Por quê?

Ao trocar o modelo durante a conversa, o assistente continuou tendo acesso ao que havia sido dito anteriormente.

Isso acontece porque o histórico da conversa é armazenado pelo Gradio e enviado novamente ao modelo escolhido a cada nova mensagem. Portanto, a memória não pertence ao Hugging Face ou ao Gemini, mas sim ao histórico mantido pela aplicação.

Quando trocamos de modelo, o código converte esse mesmo histórico para o formato esperado por cada API. No Hugging Face são utilizadas as roles user e assistant, enquanto no Gemini a resposta anterior é enviada com a role model. Dessa forma, os dois modelos conseguem continuar a conversa utilizando o mesmo contexto.


> Para parar a interface, interrompa a célula (botão de parar do Colab).

---
## Etapa 5 (Bônus): API com FastAPI

Aqui o assistente vira uma **API REST**, como no final da Aula 05.
Qualquer frontend (site, app, dispositivo IoT) poderia chamar esse endpoint.

A célula abaixo cria o arquivo `app.py`.

In [16]:
%%writefile app.py
import os
from fastapi import FastAPI
from pydantic import BaseModel
from huggingface_hub import InferenceClient

# O token e o system prompt vêm de variáveis de ambiente (nunca escreva o token no código)
client = InferenceClient(
    model="meta-llama/Llama-3.1-8B-Instruct",
    token=os.environ["HF_TOKEN"],
    provider="auto",
)
SYSTEM_PROMPT = os.environ.get("SYSTEM_PROMPT", "Você é um assistente prestativo.")

app = FastAPI()

class Pergunta(BaseModel):
    mensagem: str

@app.post("/chat")
def chat(pergunta: Pergunta):
    resposta = client.chat_completion(
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": pergunta.mensagem},
        ],
        max_tokens=300,
    )
    return {"resposta": resposta.choices[0].message.content}

Writing app.py


In [17]:
# Inicia o servidor em segundo plano, dentro do próprio Colab
!pip install fastapi uvicorn -q

import os, subprocess, time
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["SYSTEM_PROMPT"] = SYSTEM_PROMPT

servidor = subprocess.Popen(["uvicorn", "app:app", "--port", "8000"])
time.sleep(5)  # espera o servidor subir
print("Servidor rodando em http://localhost:8000")

Servidor rodando em http://localhost:8000


In [18]:
# Testa o endpoint como um frontend faria (HTTP POST com JSON)
import requests

r = requests.post(
    "http://localhost:8000/chat",
    json={"mensagem": "Como um sensor de presença pode ajudar em uma casa inteligente?"}
)

print(r.status_code)
print(r.json()["resposta"])

200
Um sensor de presença é uma ferramenta incrível para ajudar em uma casa inteligente! Ele pode detectar a presença de pessoas em uma área específica da casa, permitindo que você crie automações para atividades como:

* Ligar a luz da sala quando alguém entra;
* Aquecer ou arrefecer o ambiente quando alguém se aproxima de um determinado local;
* Ativar a música ou a TV quando alguém entra em uma sala;
* Ligar a luz de entrada quando alguém chega em casa à noite.

Por exemplo, você pode programar seu sistema de iluminação para ligar automaticamente quando alguém entra em uma sala, tornando a casa mais segura e confortável.


In [19]:
# Encerra o servidor
servidor.terminate()
print("Servidor encerrado.")

Servidor encerrado.
